# Counterfactual & Contrastive Analysis

Compare model behavior between clean and corrupted inputs to understand
what drives predictions. Find where computations diverge, measure
necessity/sufficiency of tokens, and attribute prediction differences.

This notebook covers the `irtk.counterfactual` module.

## Why This Matters

Counterfactual analysis asks: what's the minimal change to the input that would flip the model's prediction? Contrastive activation differences, necessity/sufficiency scores, and minimal-change token identification establish tight causal connections between inputs and outputs.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import counterfactual

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Contrastive Activation Diff

Where do two inputs start producing different representations?

In [ ]:
tokens_a = model.to_tokens("The Eiffel Tower is located in")
tokens_b = model.to_tokens("The Statue of Liberty is in")

result = counterfactual.contrastive_activation_diff(model, tokens_a, tokens_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result['layer_diffs'], 'bo-')
axes[0].axvline(result['divergence_layer'], color='red', alpha=0.3, label=f'Divergence layer {result["divergence_layer"]}')
axes[0].set_xlabel('Component (0=embed, 1+=layer)')
axes[0].set_ylabel('L2 Diff')
axes[0].set_title('Activation Difference by Layer')
axes[0].legend()

axes[1].plot(result['relative_diffs'], 'ro-')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Relative Diff (/ norm)')
axes[1].set_title('Relative Activation Difference')

plt.tight_layout()
plt.show()
print(f"Max diff at layer {result['max_diff_layer']}")

## 2. Counterfactual Effect by Layer

Which layers most contribute to the prediction difference?

In [ ]:
clean = model.to_tokens("The Eiffel Tower is located in")
corrupted = model.to_tokens("The Golden Gate Bridge is in")

paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

result = counterfactual.counterfactual_effect_by_layer(model, clean, corrupted, metric)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(model.cfg.n_layers), result['restoration_by_layer'], color='steelblue')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Metric Restoration')
ax.set_title(f'Layer Importance (most important: layer {result["most_important_layer"]})')
plt.tight_layout()
plt.show()

print(f"Clean metric: {result['clean_metric']:.4f}")
print(f"Corrupted metric: {result['corrupted_metric']:.4f}")

## 3. Token Necessity & Sufficiency

Which tokens are necessary (removing hurts) and sufficient (keeping alone helps)?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
result = counterfactual.token_necessity_sufficiency(model, tokens, metric)

token_strs = [tokenizer.decode([int(t)]) for t in tokens]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(len(token_strs)), result['necessity'], color='coral')
axes[0].set_yticks(range(len(token_strs)))
axes[0].set_yticklabels(token_strs)
axes[0].set_xlabel('Necessity (metric drop when removed)')
axes[0].set_title(f'Most necessary: "{token_strs[result["most_necessary"]]]}"')

axes[1].barh(range(len(token_strs)), result['sufficiency'], color='seagreen')
axes[1].set_yticks(range(len(token_strs)))
axes[1].set_yticklabels(token_strs)
axes[1].set_xlabel('Sufficiency (metric with only this token)')
axes[1].set_title(f'Most sufficient: "{token_strs[result["most_sufficient"]]]}"')

plt.tight_layout()
plt.show()

## 4. Contrastive Feature Attribution

Attribute the metric difference to attention and MLP components.

In [ ]:
tokens_a = model.to_tokens("The Eiffel Tower is located in")
tokens_b = model.to_tokens("The Golden Gate Bridge is in")

result = counterfactual.contrastive_feature_attribution(model, tokens_a, tokens_b, metric)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(model.cfg.n_layers)
w = 0.35
ax.bar(x - w/2, result['attn_attribution'], w, label='Attention', color='steelblue')
ax.bar(x + w/2, result['mlp_attribution'], w, label='MLP', color='coral')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Attribution')
ax.set_title(f'Contrastive Feature Attribution (total diff: {result["total_diff"]:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

comp_type, layer = result['most_important_component']
print(f"Most important: {comp_type} at layer {layer}")

## Summary

| Function | Purpose |
|----------|--------|
| `contrastive_activation_diff()` | Where activations diverge between two inputs |
| `minimal_change_tokens()` | Find single-token changes with largest effect |
| `counterfactual_effect_by_layer()` | Layer-wise contribution to prediction difference |
| `token_necessity_sufficiency()` | Necessity and sufficiency of each token |
| `contrastive_feature_attribution()` | Attribute differences to attn/MLP components |